# 📊 Basic ETL Example

This notebook demonstrates a simple ETL pipeline using `mlprep`.

## Scenario
We have a CSV file `data.csv` containing user data. We want to:
1. Select specific columns (`id`, `name`, `age`, `city`).
2. Filter users who are 18 years or older.
3. Save the result as a Parquet file `output.parquet`.

## 🔧 Setup

Install `mlprep-rust` package from PyPI.

In [ ]:
!pip install mlprep-rust -q

## 📁 Generate Sample Data

First, we generate sample user data with 100 rows.

In [ ]:
import pandas as pd
import numpy as np

def generate_data():
    np.random.seed(42)
    n_rows = 100
    
    data = {
        'id': range(1, n_rows + 1),
        'name': [f'User_{i}' for i in range(n_rows)],
        'age': np.random.randint(15, 60, size=n_rows),
        'city': np.random.choice(['Tokyo', 'Osaka', 'Nagoya', 'Fukuoka'], size=n_rows),
        'score': np.random.rand(n_rows) * 100
    }
    
    df = pd.DataFrame(data)
    df.to_csv('data.csv', index=False)
    print("Generated data.csv with 100 rows")
    return df

df = generate_data()
df.head(10)

## 📝 Create Pipeline Configuration

Create the `pipeline.yaml` file that defines our ETL steps.

In [ ]:
pipeline_yaml = """
name: basic_etl
inputs:
  - path: data.csv
    format: csv

steps:
  - type: select
    columns:
      - id
      - name
      - age
      - city
  - type: filter
    condition: "age >= 18"

outputs:
  - path: output.parquet
    format: parquet
"""

with open('pipeline.yaml', 'w') as f:
    f.write(pipeline_yaml.strip())

print("Created pipeline.yaml")
print(pipeline_yaml)

## 🚀 Run Pipeline

Execute the pipeline using the `mlprep` CLI command.

In [ ]:
!mlprep run pipeline.yaml

## ✅ Verify Output

Load and inspect the generated Parquet file.

In [ ]:
import pandas as pd

# Load the output parquet file
output_df = pd.read_parquet('output.parquet')

print(f"Output shape: {output_df.shape}")
print(f"\nColumns: {output_df.columns.tolist()}")
print(f"\nAge range: {output_df['age'].min()} - {output_df['age'].max()}")
print(f"\nAll ages >= 18: {(output_df['age'] >= 18).all()}")
print("\nFirst 10 rows:")
output_df.head(10)

## 📊 Summary

Compare input and output data.

In [ ]:
input_df = pd.read_csv('data.csv')

print(f"Input rows: {len(input_df)}")
print(f"Output rows: {len(output_df)}")
print(f"Rows filtered out (age < 18): {len(input_df) - len(output_df)}")
print(f"\nInput columns: {input_df.columns.tolist()}")
print(f"Output columns: {output_df.columns.tolist()}")
print(f"Columns dropped: {set(input_df.columns) - set(output_df.columns)}")